In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
import pyspark.pandas as ps
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
spark = SparkSession.builder \
    .appName("mandi-analysis") \
    .config("spark.sql.ansi.enabled", "false") \
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 10:41:23 WARN Utils: Your hostname, abhisheks-MacBook-Air-3.local, resolves to a loopback address: 127.0.0.1; using 10.140.220.117 instead (on interface en0)
26/06/02 10:41:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 10:41:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
ps_df = ps.read_csv("Agriculture_price_dataset.csv")

/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [4]:
ps_df.head()

,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
0,Maharashtra,nashik,Lasalgaon(Niphad),Wheat,Maharashtra 2189,FAQ,2172.0,2399.0,2300.0,6/6/2023
1,Maharashtra,satara,Patan,Tomato,Other,FAQ,1000.0,1500.0,1250.0,6/6/2023
2,Uttar Pradesh,mainpuri,Bewar,Potato,Local,FAQ,800.0,820.0,810.0,6/6/2023
3,Rajasthan,chittorgarh,Nimbahera,Wheat,Other,FAQ,2040.0,2668.0,2300.0,6/6/2023
4,Rajasthan,pratapgarh,Pratapgarh,Onion,Other,FAQ,476.0,1043.0,617.0,6/6/2023


In [5]:
ps_df.count()

STATE            737392
District Name    737392
Market Name      737392
Commodity        737392
Variety          737392
Grade            737392
Min_Price        737392
Max_Price        737392
Modal_Price      737392
Price Date       737392
dtype: int64

In [6]:
ps_df["Min_Price"]=ps_df["Min_Price"].astype(float)
ps_df["Max_Price"]=ps_df["Max_Price"].astype(float)
ps_df["Modal_Price"]=ps_df["Modal_Price"].astype(float)
ps_df["Price Date"] = ps.to_datetime(ps_df["Price Date"], infer_datetime_format=True)

In [7]:
ps_df.describe()

/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/namespace.py:1738: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(
/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/namespace.py:1738: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(
/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/namespace.py:1738: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A st

,Min_Price,Max_Price,Modal_Price,Price Date
count,737392.000000,737392.000000,737392.000000,737392
mean,2204.849862,2659.733824,2474.484638,2024-05-18 11:07:11.614121
min,0.000000,0.000000,0.000000,2023-06-06 00:00:00
25%,1100.000000,1500.000000,1300.000000,2023-10-20 00:00:00
50%,2000.000000,2300.000000,2150.000000,2024-05-08 00:00:00
75%,2700.000000,3205.000000,3000.000000,2024-11-28 00:00:00
max,420000.000000,480000.000000,460000.000000,2025-06-11 00:00:00
std,1920.977696,2145.250773,2023.851909,None


In [8]:
min_0 = ps_df[(ps_df['Min_Price']==0.0) & (ps_df['Max_Price']!=0.0) & (ps_df['Modal_Price']!=0.0)]
max_0 = ps_df[(ps_df['Max_Price']==0.0) & (ps_df['Min_Price']!=0.0) & (ps_df['Modal_Price']!=0.0)]
modal_0 = ps_df[(ps_df['Modal_Price']==0.0) & (ps_df['Max_Price']!=0.0) & (ps_df['Min_Price']!=0.0)]
min_max_0 = ps_df[(ps_df['Min_Price']==0.0) & (ps_df['Max_Price']==0.0) & (ps_df['Modal_Price']!=0.0)]
min_modal_0 = ps_df[(ps_df['Min_Price']==0.0) & (ps_df['Modal_Price']==0.0) & (ps_df['Max_Price']!=0.0)]
max_modal_0 = ps_df[(ps_df['Max_Price']==0.0) & (ps_df['Modal_Price']==0.0) & (ps_df['Min_Price']!=0.0)]
min_max_modal_0 = ps_df[(ps_df['Max_Price']==0.0) & (ps_df['Modal_Price']==0.0) & (ps_df['Min_Price']==0.0)]

In [9]:
print(f'\nwhole dataset has {ps_df.shape} rows;\n{min_0.shape} are with 0 min;\n{max_0.shape} are with 0 max;\n{modal_0.shape} are with 0 modal;\n{min_max_0.shape} are with min and max 0;\n{min_modal_0.shape} are with min and modal 0;\n{max_modal_0.shape} are with max and modal 0;\n{min_max_modal_0.shape} are with min, max and modal 0;')


whole dataset has (737392, 10) rows;
(65, 10) are with 0 min;
(564, 10) are with 0 max;
(1, 10) are with 0 modal;
(47, 10) are with min and max 0;
(2, 10) are with min and modal 0;
(0, 10) are with max and modal 0;
(0, 10) are with min, max and modal 0;


In [21]:
ps_df_cleanup = ps_df

In [11]:
ps_df_cleanup.head()

/Users/abhishek/explore-spark/.spark-venv/lib/python3.10/site-packages/pyspark/pandas/namespace.py:1738: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(


,STATE,District Name,Market Name,Commodity,Variety,Grade,Min_Price,Max_Price,Modal_Price,Price Date
0,Maharashtra,nashik,Lasalgaon(Niphad),Wheat,Maharashtra 2189,FAQ,2172.0,2399.0,2300.0,2023-06-06
1,Maharashtra,satara,Patan,Tomato,Other,FAQ,1000.0,1500.0,1250.0,2023-06-06
2,Uttar Pradesh,mainpuri,Bewar,Potato,Local,FAQ,800.0,820.0,810.0,2023-06-06
3,Rajasthan,chittorgarh,Nimbahera,Wheat,Other,FAQ,2040.0,2668.0,2300.0,2023-06-06
4,Rajasthan,pratapgarh,Pratapgarh,Onion,Other,FAQ,476.0,1043.0,617.0,2023-06-06


In [22]:
ps_df_cleanup[ps_df_cleanup["Min_Price"]==0.0].shape

(114, 10)

ps_df_cleanup["Min_Price"] ----> ye minimum price column nikal dega

ps_df_cleanup["Min_Price"] > 0.0 ----> te true/false ki list de dega

ps_df_cleanup[ps_df_cleanup["Min_Price"]==0.0].shape ----> ye jitne true mile hai unka count de dega

In [23]:
ps_df_cleanup[(ps_df_cleanup['Modal_Price']==0.0) & (ps_df_cleanup['Max_Price']==0.0) & (ps_df_cleanup['Min_Price']==0.0)].shape

(0, 10)

In [28]:
# drop the rows where all three Min_Price, Max_Price & Modal_Price are 0.0
# learning: among approaches 1 & 2, 2 gives expected results. approach-1 ke saath problem ye hai ki agar bas ek column bhi 0.0 milta hai to wo drop ho jara; jo ki ideally nhi hona chahiye
# approach-1:
# new_ps_df_cleanup = ps_df_cleanup[
#     (ps_df_cleanup['Min_Price']>0.0) &
#     (ps_df_cleanup['Max_Price']>0.0) &
#     (ps_df_cleanup['Modal_Price']>0.0)
# ]
# approach-2
# new_ps_df_cleanup_2 = ps_df_cleanup[
#     ~(
#         (ps_df_cleanup['Min_Price'] == 0.0) &
#         (ps_df_cleanup['Max_Price'] == 0.0) &
#         (ps_df_cleanup['Modal_Price'] == 0.0)
#     )
# ]

In [30]:
ps_df_cleanup = ps_df_cleanup[
    ~(
        (ps_df_cleanup['Min_Price'] == 0.0) &
        (ps_df_cleanup['Max_Price'] == 0.0) &
        (ps_df_cleanup['Modal_Price'] == 0.0)
    )
]

In [49]:
'''
min=0 max=0 modal!=0 --> min=modal; max=modal
min=0 max=!0 modal=0 --> min=max; modal=max
min!=0 max=0 modal=0 --> max=min; modal=min
min!=0 max!=0 modal=0 --> cannot decide most traded price here; but lets proceed with modal=min
min=!0 max=0 modal!=0 --> max=modal <- ideal; but do max=max(min, modal)
min=0 max=!0 modal!=0 --> min=min(max, modal)
'''

In [ ]:
spark.stop()